In [5]:
!pip install requests beautifulsoup4 ollama

Defaulting to user installation because normal site-packages is not writeable
  Using cached ollama-0.6.1-py3-none-any.whl.metadata (4.3 kB)
Using cached ollama-0.6.1-py3-none-any.whl (14 kB)



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\alice\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
import requests
from bs4 import BeautifulSoup
import time
import json
import ollama

In [9]:
def get_links_from_archive(page_num=1):
    """아카이브 페이지에서 뉴스 링크 수집"""
    url = f"https://www.therundown.ai/archive?page={page_num}"
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        print(f"Fetching archive page: {url}")
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code != 200:
            print(f"Page {page_num} returned status {response.status_code}. Stopping.")
            return []
            
        soup = BeautifulSoup(response.content, 'html.parser')
        
        links = []
        for a in soup.find_all('a', href=True):
            href = a['href']
            if '/p/' in href and 'archive' not in href:
                full_link = href if href.startswith('http') else f"https://www.therundown.ai{href}"
                if full_link not in links:
                    links.append(full_link)
        
        if not links:
            print(f"No links found on page {page_num}. Ending collection.")
            return []
            
        print(f"Found {len(links)} links on page {page_num}")
        return links
    except Exception as e:
        print(f"Error fetching archive page {page_num}: {e}")
        return []

In [10]:
a = get_links_from_archive(page_num=1)
a

Fetching archive page: https://www.therundown.ai/archive?page=1
Found 12 links on page 1


['https://www.therundown.ai/p/open-source-ai-crushes-elite-math-exam',
 'https://www.therundown.ai/p/microsofts-cancer-mapping-ai',
 'https://www.therundown.ai/p/openai-reveals-whos-winning-with-ai-at-work',
 'https://www.therundown.ai/p/poetiq-cracks-major-reasoning-benchmark',
 'https://www.therundown.ai/p/anthropic-puts-claude-in-the-interviewer-chair',
 'https://www.therundown.ai/p/anthropic-and-openai-s-ipo-showdown',
 'https://www.therundown.ai/p/openais-code-red-scramble',
 'https://www.therundown.ai/p/deepseek-strikes-again',
 'https://www.therundown.ai/p/ai-cracks-30-year-math-problem',
 'https://www.therundown.ai/p/deepseek-returns-with-an-imo-crushing-ai',
 'https://www.therundown.ai/p/karpathys-advice-for-the-ai-classroom',
 'https://www.therundown.ai/p/ilya-sutskever-breaks-silence-on-ai-future']

In [ ]:
def scrape_article(url):
    """URL에서 기사 내용 스크래핑"""
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
            
        for script in soup(["script", "style", "nav", "footer"]):
            script.decompose()

        for a in soup.find_all('a', href=True):
            if a.get_text(strip=True):
                a.replace_with(f" {a.get_text(strip=True)} ({a['href']}) ")

        text = soup.get_text(separator='\n', strip=True)
        
        # 날짜 추출 및 관련 콘텐츠 추출
        text = extract_relevant_content(text)
        
        return {
            "text": text, 
            "url": url
        }
    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return None


def extract_date(text):
    """텍스트에서 날짜 추출 (예: December 11, 2025)"""
    import re
    
    # 월 이름 패턴
    months = r'(?:January|February|March|April|May|June|July|August|September|October|November|December)'
    
    # 날짜 패턴: Month DD, YYYY
    date_pattern = rf'({months}\s+\d{{1,2}},\s+\d{{4}})'
    
    match = re.search(date_pattern, text)
    if match:
        return match.group(1)
    
    return None


def extract_relevant_content(text):
    """LATEST DEVELOPMENTS 이후 내용에서 광고/불필요 섹션 제외하고 추출"""
    
    # 먼저 날짜 추출 (LATEST DEVELOPMENTS 이전에서)
    article_date = extract_date(text)
    
    # LATEST DEVELOPMENTS 이후 내용만 가져오기
    start_marker = "LATEST DEVELOPMENTS"
    start_idx = text.find(start_marker)
    if start_idx != -1:
        text = text[start_idx:]
    
    # 제외할 섹션 시작 키워드들
    exclude_markers = [
        "PRESENTED BY",
        "TOGETHER WITH", 
        "AI TRAINING",
        "Trending AI Tools",
        "Community AI workflows",
        "Highlights: News, Guides & Events",
        "That's it for today!"
    ]
    
    # 포함할 섹션 (제외 마커보다 우선)
    include_markers = [
        "Everything else in AI today"
    ]
    
    # 텍스트를 줄 단위로 분리
    lines = text.split('\n')
    
    result_lines = []
    
    # 날짜가 있으면 맨 앞에 추가
    if article_date:
        result_lines.append(f"Date: {article_date}")
        result_lines.append("")  # 빈 줄 추가
    
    skip_section = False
    
    for line in lines:
        stripped_line = line.strip()
        
        # 포함할 섹션 시작 체크 (우선 처리)
        should_include = False
        for marker in include_markers:
            if marker in stripped_line:
                should_include = True
                skip_section = False
                break
        
        if should_include:
            result_lines.append(line)
            continue
        
        # 제외할 섹션 시작 체크
        should_skip = False
        for marker in exclude_markers:
            if stripped_line.startswith(marker) or marker in stripped_line:
                should_skip = True
                skip_section = True
                break
        
        if should_skip:
            continue
            
        # 새로운 주요 섹션 시작 시 skip 해제 (대문자로 된 섹션 헤더)
        if stripped_line.isupper() and len(stripped_line) > 3 and ' ' in stripped_line:
            is_exclude = any(marker in stripped_line for marker in exclude_markers)
            if not is_exclude:
                skip_section = False
        
        if not skip_section:
            result_lines.append(line)
    
    return '\n'.join(result_lines)

def analyze_with_ollama_english(article_data, model="qwen2.5:7b"):
    """Ollama로 영어 요약 생성"""
    
    prompt = f"""# System Role: AI News Analyst

You are an expert AI News Analyst. Your task is to extract and structure AI news from the provided text into JSON format.

## Input Information
- Source URL: {article_data['url']}
- Article Content:
{article_data['text']}

## Your Task
Extract ALL individual news items from the content above and output them as a JSON array.

## Rules for Each News Item

1. **title**: Rewrite the original title in a different way while keeping the same meaning (paraphrase it creatively)

2. **summary**: Write a 2-3 sentence summary explaining the key points of the news

3. **source**: Extract the publisher/company name mentioned in the article (e.g., "Nous Research", "Microsoft", "Core Devices")

4. **sourceUrl**: Use the URL that appears in the article content for that specific news item. Look for URLs in parentheses like (https://...)

5. **publishedDate**: Use the date from the beginning of the content (look for "Date:" at the top)

6. **likes, shareCount, popularityScore**: Set all to 0

7. **categories**: Select ALL that apply from this list:
   - "Business"
   - "Tech/AI"
   - "Healthcare/Science"
   - "Entertainment/Creative"
   - "Education"
   - "Society/Policy"
   - "Hardware"
   - "Lifestyle"

8. **productServices**: Select ALL that apply from this list:
   - "Text AI"
   - "Image AI"
   - "Video AI"
   - "Voice AI"
   - "Agent AI"
   - "Automation AI"
   - "Multimodal AI"
   - "Vibe Coding AI"
   - "Robotics"

9. **coreElements**: Select ALL that apply from this list:
   - "Data"
   - "Algorithm"
   - "Compute"
   - "Safety/Ethics"

10. **searchKeywords**: Extract 3-5 relevant keywords in lowercase

## Output Format
Output ONLY a valid JSON array with NO markdown code blocks, NO extra text.

Example format:
[
  {{
    "title": "Paraphrased title here",
    "summary": "2-3 sentence summary here",
    "source": "Publisher Name",
    "sourceUrl": "https://example.com/article",
    "publishedDate": "December 11, 2025",
    "likes": 0,
    "shareCount": 0,
    "popularityScore": 0,
    "categories": ["Tech/AI", "Business"],
    "productServices": ["Text AI", "Agent AI"],
    "coreElements": ["Algorithm", "Data"],
    "searchKeywords": ["keyword1", "keyword2", "keyword3"]
  }}
]

Now extract ALL news items and output the JSON array:"""

    try:
        response = ollama.generate(
            model=model,
            prompt=prompt,
            options={
                "temperature": 0.1,
                "num_predict": 4096
            }
        )
        
        json_str = response['response'].strip()
        
        # Markdown 제거
        if json_str.startswith('```json'):
            json_str = json_str[7:]
        if json_str.startswith('```'):
            json_str = json_str[3:]
        if json_str.endswith('```'):
            json_str = json_str[:-3]
        json_str = json_str.strip()
        
        # JSON 파싱
        result = json.loads(json_str)
        return result if isinstance(result, list) else [result]
        
    except json.JSONDecodeError as e:
        print(f"JSON parsing error: {e}")
        print(f"Raw response: {response['response'][:500]}")
        return None
    except Exception as e:
        print(f"Ollama English analysis error: {e}")
        return None


In [ ]:
import requests
from bs4 import BeautifulSoup
import ollama
import json
import re


def scrape_article(url):
    """URL에서 기사 내용 스크래핑"""
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
            
        for script in soup(["script", "style", "nav", "footer"]):
            script.decompose()

        for a in soup.find_all('a', href=True):
            if a.get_text(strip=True):
                a.replace_with(f" {a.get_text(strip=True)} ({a['href']}) ")

        text = soup.get_text(separator='\n', strip=True)
        
        # 날짜 추출 및 관련 콘텐츠 추출
        text = extract_relevant_content(text)
        
        return {
            "text": text, 
            "url": url
        }
    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return None


def extract_date(text):
    """텍스트에서 날짜 추출 (예: December 11, 2025)"""
    months = r'(?:January|February|March|April|May|June|July|August|September|October|November|December)'
    date_pattern = rf'({months}\s+\d{{1,2}},\s+\d{{4}})'
    
    match = re.search(date_pattern, text)
    if match:
        return match.group(1)
    
    return None


def extract_relevant_content(text):
    """LATEST DEVELOPMENTS 이후 내용에서 광고/불필요 섹션 제외하고 추출"""
    
    # 먼저 날짜 추출 (LATEST DEVELOPMENTS 이전에서)
    article_date = extract_date(text)
    
    # LATEST DEVELOPMENTS 이후 내용만 가져오기
    start_marker = "LATEST DEVELOPMENTS"
    start_idx = text.find(start_marker)
    if start_idx != -1:
        text = text[start_idx:]
    
    # 제외할 섹션 시작 키워드들
    exclude_markers = [
        "PRESENTED BY",
        "TOGETHER WITH", 
        "AI TRAINING",
        "Trending AI Tools",
        "Community AI workflows",
        "Highlights: News, Guides & Events",
        "That's it for today!"
    ]
    
    # 포함할 섹션 (제외 마커보다 우선)
    include_markers = [
        "Everything else in AI today"
    ]
    
    # 텍스트를 줄 단위로 분리
    lines = text.split('\n')
    
    result_lines = []
    
    # 날짜가 있으면 맨 앞에 추가
    if article_date:
        result_lines.append(f"Date: {article_date}")
        result_lines.append("")  # 빈 줄 추가
    
    skip_section = False
    
    for line in lines:
        stripped_line = line.strip()
        
        # 포함할 섹션 시작 체크 (우선 처리)
        should_include = False
        for marker in include_markers:
            if marker in stripped_line:
                should_include = True
                skip_section = False
                break
        
        if should_include:
            result_lines.append(line)
            continue
        
        # 제외할 섹션 시작 체크
        should_skip = False
        for marker in exclude_markers:
            if stripped_line.startswith(marker) or marker in stripped_line:
                should_skip = True
                skip_section = True
                break
        
        if should_skip:
            continue
            
        # 새로운 주요 섹션 시작 시 skip 해제 (대문자로 된 섹션 헤더)
        if stripped_line.isupper() and len(stripped_line) > 3 and ' ' in stripped_line:
            is_exclude = any(marker in stripped_line for marker in exclude_markers)
            if not is_exclude:
                skip_section = False
        
        if not skip_section:
            result_lines.append(line)
    
    return '\n'.join(result_lines)


def analyze_with_ollama_english(article_data, model="qwen2.5:7b"):
    """Ollama로 영어 요약 생성
    
    Args:
        article_data: scrape_article()의 반환값 {"text": str, "url": str}
        model: 사용할 Ollama 모델명
    
    Returns:
        list: JSON 형식의 뉴스 아이템 리스트
    """
    
    # article_data가 dict인지 확인
    if isinstance(article_data, dict):
        text = article_data.get('text', '')
        source_url = article_data.get('url', '')
    elif isinstance(article_data, str):
        # 문자열로 직접 전달된 경우
        text = article_data
        source_url = ''
    else:
        print(f"Invalid article_data type: {type(article_data)}")
        return None
    
    prompt = f"""# System Role: AI News Analyst

                    You are an expert AI News Analyst. Your task is to extract and structure AI news from the provided text into JSON format.

                    ## Input Information
                    - Article Content:{text}
                    - Source URL: {source_url}

                    ## Your Task
                    Extract ALL individual news items from the content above and output them as a JSON array.

                    ## Content Structure
                    The content has two types of articles:

                    **Type 1: Main Articles (with full details)**
                    These appear with section headers like "NOUS RESEARCH", "AI RESEARCH", "AI WEARABLES", etc.
                    - They have detailed descriptions with "The details:", "Why it matters:" sections

                    **Type 2: Brief Articles (under "Everything else in AI today")**
                    These are short news items that appear after "Everything else in AI today" section.
                    - Format: "Company/Topic [verb] (URL) description"
                    - Example: "Chinese AI startup DeepSeek is reportedly developing (https://...) its next AI model..."
                    - Extract EACH brief item as a separate news entry
                    - The URL immediately following the title/verb is the sourceUrl for that specific article

                    ## Rules for Each News Item

                    1. **title**: Rewrite the original title in a different way while keeping the same meaning (paraphrase it creatively)

                    2. **summary**: Write a 2-3 sentence summary explaining the key points of the news

                    3. **source**: Extract the publisher/company name mentioned in the article (e.g., "Nous Research", "Microsoft", "DeepSeek", "McDonald's", "Amazon")

                    4. **sourceUrl**: 
                    - For main articles: Use the URL in parentheses that appears right after the article title
                    - For "Everything else in AI today" items: Use the URL that appears immediately after each news item's title/description verb (e.g., "reportedly developing (https://...)" - use that https URL)
                    - DO NOT use the "Everything else in AI today" section URL itself

                    5. **publishedDate**: Use the date from the beginning of the content (look for "Date:" at the top). Format: "Month DD, YYYY"

                    6. **likes, shareCount, popularityScore**: Set all to 0

                    7. **categories**: Select ALL that apply from this list:
                    - "Business"
                    - "Tech/AI"
                    - "Healthcare/Science"
                    - "Entertainment/Creative"
                    - "Education"
                    - "Society/Policy"
                    - "Hardware"
                    - "Lifestyle"

                    8. **productServices**: Select ALL that apply from this list:
                    - "Text AI"
                    - "Image AI"
                    - "Video AI"
                    - "Voice AI"
                    - "Agent AI"
                    - "Automation AI"
                    - "Multimodal AI"
                    - "Vibe Coding AI"
                    - "Robotics"

                    9. **coreElements**: Select ALL that apply from this list:
                    - "Data"
                    - "Algorithm"
                    - "Compute"
                    - "Safety/Ethics"

                    10. **searchKeywords**: Extract 3-5 relevant keywords in lowercase

                    ## Output Format
                    Output ONLY a valid JSON array with NO markdown code blocks, NO extra text.

                    Each news item should follow this exact format:
                    [
                    {{
                        "title": "Paraphrased title here",
                        "summary": "2-3 sentence summary here",
                        "source": "Publisher Name",
                        "sourceUrl": "https://example.com/article",
                        "publishedDate": "December 11, 2025",
                        "likes": 0,
                        "shareCount": 0,
                        "popularityScore": 0,
                        "categories": ["Tech/AI", "Business"],
                        "productServices": ["Text AI", "Agent AI"],
                        "coreElements": ["Algorithm", "Data"],
                        "searchKeywords": ["keyword1", "keyword2", "keyword3"]
                    }}
                    ]

                    IMPORTANT: Include BOTH main articles AND all brief articles from "Everything else in AI today" section. Each brief item is a separate JSON object.

                    Now extract ALL news items and output the JSON array:"""

    try:
        response = ollama.generate(
            model=model,
            prompt=prompt,
            options={
                "temperature": 0.1,
                "num_predict": 4096
            }
        )
        
        json_str = response['response'].strip()
        
        # Markdown 제거
        if json_str.startswith('```json'):
            json_str = json_str[7:]
        if json_str.startswith('```'):
            json_str = json_str[3:]
        if json_str.endswith('```'):
            json_str = json_str[:-3]
        json_str = json_str.strip()
        
        # JSON 파싱
        result = json.loads(json_str)
        return result if isinstance(result, list) else [result]
        
    except json.JSONDecodeError as e:
        print(f"JSON parsing error: {e}")
        print(f"Raw response: {response['response'][:500]}")
        return None
    except Exception as e:
        print(f"Ollama English analysis error: {e}")
        return None



In [33]:
url = "https://www.therundown.ai/p/open-source-ai-crushes-elite-math-exam"
article_data = scrape_article(url)
article_data

{'text': "Date: December 11, 2025\n\nLATEST DEVELOPMENTS\nNOUS RESEARCH\n📈\nNous Research's AI takes on elite math exam (https://x.com/NousResearch/status/1998536543565127968?s=20&utm_source=www.therundown.ai&utm_medium=referral&utm_campaign=open-source-ai-crushes-elite-math-exam)\nImage source: Nano Banana Pro / The Rundown\nThe Rundown:\nNous Research just\nopen-sourced (https://x.com/NousResearch/status/1998536543565127968?s=20&utm_source=www.therundown.ai&utm_medium=referral&utm_campaign=open-source-ai-crushes-elite-math-exam)\nNomos 1, a new 30B parameter reasoning system that scored 87 out of 120 on the 2025 Putnam Contest — crushing rivals like Qwen 3 on one of the most prestigious collegiate math competitions.\nThe details:\nThe system uses a two-phase approach: AI ‘workers’ solve and self-critique responses, with a tournament-style bracket then selecting the best submission.\nNomos’ score would have placed second among nearly 4,000 human competitors last year, with the model e

In [36]:
print(article_data['text'])

Date: December 11, 2025

LATEST DEVELOPMENTS
NOUS RESEARCH
📈
Nous Research's AI takes on elite math exam (https://x.com/NousResearch/status/1998536543565127968?s=20&utm_source=www.therundown.ai&utm_medium=referral&utm_campaign=open-source-ai-crushes-elite-math-exam)
Image source: Nano Banana Pro / The Rundown
The Rundown:
Nous Research just
open-sourced (https://x.com/NousResearch/status/1998536543565127968?s=20&utm_source=www.therundown.ai&utm_medium=referral&utm_campaign=open-source-ai-crushes-elite-math-exam)
Nomos 1, a new 30B parameter reasoning system that scored 87 out of 120 on the 2025 Putnam Contest — crushing rivals like Qwen 3 on one of the most prestigious collegiate math competitions.
The details:
The system uses a two-phase approach: AI ‘workers’ solve and self-critique responses, with a tournament-style bracket then selecting the best submission.
Nomos’ score would have placed second among nearly 4,000 human competitors last year, with the model earning eight perfect pr

In [34]:
results = analyze_with_ollama_english(article_data)

In [35]:
results

[{'title': 'Open-Source AI Surpasses Elite Math Exam',
  'summary': "Nous Research's open-source AI system, Nomos 1, scored 87 out of 120 on the prestigious Putnam Contest, significantly outperforming other models like Qwen 3. The system uses a two-phase approach involving AI workers and self-critique to generate top-notch solutions.",
  'source': 'Nous Research',
  'sourceUrl': 'https://x.com/NousResearch/status/1998536543565127968?s=20&utm_source=www.therundown.ai&utm_medium=referral&utm_campaign=open-source-ai-crushes-elite-math-exam',
  'publishedDate': 'December 11, 2025',
  'likes': 0,
  'shareCount': 0,
  'popularityScore': 0,
  'categories': ['Tech/AI', 'Education'],
  'productServices': ['Text AI'],
  'coreElements': ['Algorithm', 'Compute'],
  'searchKeywords': ['ai', 'math exam', 'open-source']},
 {'title': 'Microsoft Analyzes Copilot Usage Patterns',
  'summary': 'Microsoft released a study based on 37.5 million Copilot conversations, revealing that health and wellness quer

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import json
import ollama

def get_links_from_archive(page_num=1):
    """아카이브 페이지에서 뉴스 링크 수집"""
    url = f"https://www.therundown.ai/archive?page={page_num}"
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        print(f"Fetching archive page: {url}")
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code != 200:
            print(f"Page {page_num} returned status {response.status_code}. Stopping.")
            return []
            
        soup = BeautifulSoup(response.content, 'html.parser')
        
        links = []
        for a in soup.find_all('a', href=True):
            href = a['href']
            if '/p/' in href and 'archive' not in href:
                full_link = href if href.startswith('http') else f"https://www.therundown.ai{href}"
                if full_link not in links:
                    links.append(full_link)
        
        if not links:
            print(f"No links found on page {page_num}. Ending collection.")
            return []
            
        print(f"Found {len(links)} links on page {page_num}")
        return links
    except Exception as e:
        print(f"Error fetching archive page {page_num}: {e}")
        return []


def scrape_article(url):
    """URL에서 기사 내용 스크래핑"""
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        
        title = soup.find('h1').get_text(strip=True) if soup.find('h1') else (soup.title.string if soup.title else "No Title")
        
        date_text = ""
        time_tag = soup.find('time')
        if time_tag and time_tag.has_attr('datetime'):
            date_text = time_tag['datetime']
            
        for script in soup(["script", "style", "nav", "footer"]):
            script.decompose()

        for a in soup.find_all('a', href=True):
            if a.get_text(strip=True):
                a.replace_with(f" {a.get_text(strip=True)} ({a['href']}) ")

        text = soup.get_text(separator='\n', strip=True)
        
        return {
            "title": title,
            "date": date_text,
            "text": text[:8000],  # 토큰 제한 방지
            "url": url
        }
    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return None


def analyze_with_ollama_english(article_data, model="qwen2.5:7b"):
    """Ollama로 영어 요약 생성"""
    
    prompt = f"""# System Role: AI News Analyst & Web Scraper

        You are an expert AI News Analyst. Your task is to extract, analyze, and structure AI news from raw webpage text into a specific JSON format in **Korean**.

        ## 1. Extraction Strategy (Strict Execution Order)
        **1. Global Date Extraction:** - First, find the main publication date at the top of the webpage or input text. 
        - Use this extracted date for the `publishedDate` field for all articles (convert to ISO 8601 format).

        **2. Section Targeting ("LATEST DEVELOPMENTS"):**
        - Locate the header **"LATEST DEVELOPMENTS"**.
        - **Ignore** all content before this header (except for the Global Date).
        - **Process EVERY single article** found after the "LATEST DEVELOPMENTS" header.

        **3. Exclusion Rules:**
        - Even after the target header, exclude sections starting with `PRESENTED BY`, `TOGETHER WITH`, `AI TRAINING`, or `Trending AI Tools`.

        ## 2. Content Generation (Korean Output)
        For **each** article extracted after "LATEST DEVELOPMENTS":

        * **title:** Rewrite the English title into a natural, compelling **Korean** title (paraphrase the meaning).
        * **summary:** Write a clear, factual summary in **Korean** (2-3 sentences).
        * **source:** Extract the publisher name (e.g., TechCrunch, The Verge).
        * **sourceUrl:** Extract the direct hyperlink (URL) to the article.
        * **searchKeywords:** Extract 3-5 keywords (lower case).
        * **publishedDate:** Use the **Global Date** extracted in Step 1.
        * **Metrics:** Set `likes`, `shareCount`, and `popularityScore` to `0` as defaults.

        ## 3. Categorization Rules (Strict Match)
        Classify each article using **ONLY** the exact strings provided below.

        **A. Categories (`categories`)**
        > `["비즈니스", "테크/AI", "헬스케어/과학", "엔터/창작", "교육", "사회/정책", "하드웨어", "라이프스타일"]`

        **B. Core Elements (`coreElements`)**
        > `["데이터", "알고리즘", "컴퓨트", "안전/윤리"]`

        **C. Product Services (`productServices`)**
        > `["텍스트 AI", "이미지 AI", "동영상 AI", "음성 AI", "에이전트 AI", "자동화 AI", "멀티모달 AI", "바이브 코딩 AI", "로봇"]`

        ## 4. Output Format
        Output the result as a single valid **JSON Array** inside a code block.
        For each news item, output this exact JSON format:
        {{
            "title": "Original English title",
            "summary": "2-3 sentence summary in English",
            "source": "Publisher name (e.g., TechCrunch)",
            "sourceUrl": "Direct URL to the article",
            "publishedDate": "December 11, 2025",
            "likes": 0,
            "shareCount": 0,
            "popularityScore": 0,
            "categories": ["Choose from: Business, Tech/AI, Healthcare/Science, Entertainment/Creative, Education, Society/Policy, Hardware, Lifestyle"],
            "productServices": ["Choose from: Text AI, Image AI, Video AI, Voice AI, Agent AI, Automation AI, Multimodal AI, Vibe Coding AI, Robotics"],
            "coreElements": ["Choose from: Data, Algorithm, Compute, Safety/Ethics"],
            "searchKeywords": ["keyword1", "keyword2", "keyword3"]
        }}

        Rules:
        - Output ONLY valid JSON array
        - NO markdown code blocks
        - Extract ALL news items from the article
        - Skip sections: PRESENTED BY, TOGETHER WITH, AI TRAINING, Trending AI Tools

        Article Content:
        Source URL: {article_data['url']}
        Body:
        {article_data['text']}

        Output JSON array:"""

    try:
        response = ollama.generate(
            model=model,
            prompt=prompt,
            options={
                "temperature": 0.1,
                "num_predict": 4096
            }
        )
        
        json_str = response['response'].strip()
        
        # Markdown 제거
        if json_str.startswith('```json'):
            json_str = json_str[7:]
        if json_str.startswith('```'):
            json_str = json_str[3:]
        if json_str.endswith('```'):
            json_str = json_str[:-3]
        json_str = json_str.strip()
        
        # JSON 파싱
        result = json.loads(json_str)
        return result if isinstance(result, list) else [result]
        
    except json.JSONDecodeError as e:
        print(f"JSON parsing error: {e}")
        print(f"Raw response: {response['response'][:500]}")
        return None
    except Exception as e:
        print(f"Ollama English analysis error: {e}")
        return None


def translate_to_korean(english_articles, model="qwen2.5:7b"):
    """영어 기사를 한국어로 번역"""
    
    korean_articles = []
    
    for i, article in enumerate(english_articles):
        print(f"  Translating article {i+1}/{len(english_articles)}...")
        
        prompt = f"""Translate this news article data to Korean.

Input:
- Title: {article.get('title', '')}
- Summary: {article.get('summary', '')}
- Categories: {article.get('categories', [])}
- Product Services: {article.get('productServices', [])}
- Core Elements: {article.get('coreElements', [])}

Output ONLY valid JSON with these exact keys:
{{
    "title": "Korean translated title",
    "summary": "Korean translated summary (2-3 sentences)",
    "source": "{article.get('source', '')}",
    "sourceUrl": "{article.get('sourceUrl', '')}",
    "publishedDate": "{article.get('publishedDate', '')}",
    "likes": 0,
    "shareCount": 0,
    "popularityScore": 0,
    "categories": ["Use Korean: 비즈니스, 테크/AI, 헬스케어/과학, 엔터/창작, 교육, 사회/정책, 하드웨어, 라이프스타일"],
    "productServices": ["Use Korean: 텍스트 AI, 이미지 AI, 동영상 AI, 음성 AI, 에이전트 AI, 자동화 AI, 멀티모달 AI, 바이브 코딩 AI, 로봇"],
    "coreElements": ["Use Korean: 데이터, 알고리즘, 컴퓨트, 안전/윤리"],
    "searchKeywords": {article.get('searchKeywords', [])}
}}

NO markdown, NO explanation. Output JSON only:"""

        try:
            response = ollama.generate(
                model=model,
                prompt=prompt,
                options={
                    "temperature": 0.1,
                    "num_predict": 2048
                }
            )
            
            json_str = response['response'].strip()
            
            # Markdown 제거
            if json_str.startswith('```json'):
                json_str = json_str[7:]
            if json_str.startswith('```'):
                json_str = json_str[3:]
            if json_str.endswith('```'):
                json_str = json_str[:-3]
            json_str = json_str.strip()
            
            korean_article = json.loads(json_str)
            korean_articles.append(korean_article)
            
        except Exception as e:
            print(f"  Translation error for article {i+1}: {e}")
            # 번역 실패시 원본 유지
            korean_articles.append(article)
        
        time.sleep(1)  # Rate limiting
    
    return korean_articles


def process_all_urls(links, model="qwen2.5:7b"):
    """모든 URL 처리 - 영어 요약 + 한국어 번역"""
    
    all_english = []
    all_korean = []
    
    print(f"\n{'='*60}")
    print(f"Processing {len(links)} links with model: {model}")
    print(f"{'='*60}\n")
    
    for i, link in enumerate(links):
        print(f"\n[{i+1}/{len(links)}] Processing: {link}")
        
        # 1. 기사 스크래핑
        article_data = scrape_article(link)
        if not article_data:
            print("  -> Scraping failed, skipping...")
            continue
        
        # 2. 영어 요약 생성
        print("  -> Generating English summary...")
        english_articles = analyze_with_ollama_english(article_data, model)
        
        if not english_articles:
            print("  -> English analysis failed, skipping...")
            continue
        
        print(f"  -> Extracted {len(english_articles)} news items")
        all_english.extend(english_articles)
        
        # 3. 한국어 번역
        print("  -> Translating to Korean...")
        korean_articles = translate_to_korean(english_articles, model)
        all_korean.extend(korean_articles)
        
        print(f"  -> Done! Total so far: EN={len(all_english)}, KR={len(all_korean)}")
        
        time.sleep(2)  # Rate limiting between pages
    
    return all_english, all_korean

In [7]:
if __name__ == "__main__":
    
    # ============ 설정 ============
    START_PAGE = 1      # 시작 페이지
    END_PAGE = 2        # 끝 페이지
    MODEL = "qwen2.5:7b"  # 사용할 Ollama 모델
    # ==============================
    
    print(f"\n{'#'*60}")
    print(f"# AI News Scraper with Ollama")
    print(f"# Pages: {START_PAGE} to {END_PAGE}")
    print(f"# Model: {MODEL}")
    print(f"{'#'*60}\n")
    
    # 1. 링크 수집
    all_links = []
    for page_num in range(START_PAGE, END_PAGE + 1):
        print(f"{page_num} 페이지 수집 중...")
        
        page_links = get_links_from_archive(page_num)
        
        if not page_links:
            print(f"{page_num} 페이지에 링크가 없습니다. 수집을 조기 종료합니다.")
            break
        
        all_links.extend(page_links)
        time.sleep(1)
    
    # 중복 제거
    unique_links = list(set(all_links))
    print(f"\n--- Total unique links found: {len(unique_links)} ---\n")
    
    # 2. 모든 URL 처리
    if unique_links:
        english_articles, korean_articles = process_all_urls(unique_links, MODEL)
        
        # 3. 결과 저장
        print(f"\n{'='*60}")
        print(f"RESULTS")
        print(f"{'='*60}")
        print(f"Total English articles: {len(english_articles)}")
        print(f"Total Korean articles: {len(korean_articles)}")
        
        # 영어 JSON 저장
        en_filename = f'news_english_p{START_PAGE}_p{END_PAGE}.json'
        with open(en_filename, 'w', encoding='utf-8') as f:
            json.dump(english_articles, f, ensure_ascii=False, indent=2)
        print(f"\n✅ Saved English: {en_filename}")
        
        # 한국어 JSON 저장
        kr_filename = f'news_korean_p{START_PAGE}_p{END_PAGE}.json'
        with open(kr_filename, 'w', encoding='utf-8') as f:
            json.dump(korean_articles, f, ensure_ascii=False, indent=2)
        print(f"✅ Saved Korean: {kr_filename}")
        
    else:
        print("No links found to process.")



############################################################
# AI News Scraper with Ollama
# Pages: 1 to 2
# Model: qwen2.5:7b
############################################################

1 페이지 수집 중...
Fetching archive page: https://www.therundown.ai/archive?page=1
Found 12 links on page 1
2 페이지 수집 중...
Fetching archive page: https://www.therundown.ai/archive?page=2
Found 12 links on page 2

--- Total unique links found: 24 ---


Processing 24 links with model: qwen2.5:7b


[1/24] Processing: https://www.therundown.ai/p/deepseek-strikes-again
  -> Generating English summary...
  -> Extracted 1 news items
  -> Translating to Korean...
  Translating article 1/1...
  Translation error for article 1: Expecting value: line 13 column 24 (char 613)
  -> Done! Total so far: EN=1, KR=1

[2/24] Processing: https://www.therundown.ai/p/openais-code-red-scramble
  -> Generating English summary...
  -> Extracted 1 news items
  -> Translating to Korean...
  Translating article 1/1...
  Translation 